In [ ]:
# Change path
import os, sys
repo_path = os.path.abspath(os.path.join(os.path.abspath(''), '..'))
assert os.path.basename(repo_path) == 'textdd', "Wrong parent folder. Please change to 'textdd'"
if sys.path[0] != repo_path:
    sys.path.insert(0, repo_path)

from functools import partial
from typing import Optional
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import (
    plot_from_csv_callback,
    aggregate_results_from_csv_to_dict_callback,
    list_contents_pre_callback,
    list_results_from_csv_pre_callback,
    load_preset_plt,
    walk_contents,
)

def load_preset_plt():
    """Config preset parameters for matplotlib."""
    plt.rcParams.update({
        'figure.titlesize': 16,
        'axes.titlesize':   14,
        'axes.labelsize':   14,
        'font.size':        12,
        'xtick.labelsize':  12,
        'ytick.labelsize':  12,
        'legend.fontsize':  12,
        'lines.linewidth':  1,
    })
load_preset_plt()

In [ ]:
REPORTED = Path("../results/reported")

no_val_acc = []
for csv in sorted(REPORTED.rglob("*.csv")):
    cols = pd.read_csv(csv, nrows=0).columns.tolist()
    if "val_acc" not in cols:
        no_val_acc.append((str(csv.relative_to(REPORTED)), cols[-3:]))

print(f"{len(no_val_acc)} CSVs missing val_acc:")
for path, tail in no_val_acc:
    print(f"  {path}  →  last cols: {tail}")

In [ ]:
REPORTED = Path("../results/reported")

def load_dir_stats(dirpath: Path, metric: str = "val_acc", target: str = "last") -> dict | None:
    csvs = sorted(dirpath.glob("*.csv"))
    if not csvs:
        return None
    vals = []
    for csv in csvs:
        col = pd.read_csv(csv)[metric].dropna()
        if target == "max":
            vals.append(col.max())
        elif target == "min":
            vals.append(col.min())
        else:
            vals.append(col.iloc[-1])
    return {"mean": float(np.mean(vals)) * 100, "std": float(np.std(vals)) * 100, "n": len(vals)}

def fmt(stats: dict | None) -> str:
    if stats is None:
        return "—"
    return f"{stats['mean']:.2f} ± {stats['std']:.2f}"


In [ ]:

# main-cls: rows=(dataset, setting), cols=classifier
CLS_DATASETS  = ["agnews", "imdb", "sst2"]
CLS_MODELS    = ["logistic", "textcnn", "textrnn", "albert"]
CLS_SETTINGS  = ["base", "rand", "randeda", "take"]

rows, index = [], []
for ds in CLS_DATASETS:
    for setting in CLS_SETTINGS:
        row = []
        for mdl in CLS_MODELS:
            d = REPORTED / "main-cls" / f"clf-{ds}-{mdl}" / setting
            row.append(fmt(load_dir_stats(d)))
        rows.append(row)
        index.append((ds, setting))

df_main_cls = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(index, names=["dataset", "setting"]), columns=CLS_MODELS)
df_main_cls


In [ ]:
# main-nli: rows=(dataset, setting), cols=classifier
# NLI_DATASETS = ["mnlim", "qnli"]
NLI_DATASETS = ["mnlim", "qnli", "qqp"]
NLI_MODELS   = ["siamlog", "albert"]
NLI_SETTINGS = ["base", "rand", "randeda", "take"]

rows, index = [], []
for ds in NLI_DATASETS:
    for setting in NLI_SETTINGS:
        row = []
        for mdl in NLI_MODELS:
            d = REPORTED / "main-nli" / f"clf-{ds}-{mdl}" / setting
            row.append(fmt(load_dir_stats(d)))
        rows.append(row)
        index.append((ds, setting))

df_main_nli = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(index, names=["dataset", "setting"]), columns=NLI_MODELS)
df_main_nli


In [ ]:
# abla-components: rows=setting, cols=(dataset, classifier)
ABLA_SETTINGS = ["rand", "kmeans-base", "kmeans-cond", "take-donttake", "take"]
ABLA_COLS = [  # (dataset, model, eval_subdir)
    ("agnews", "logistic","eval_cls"),
    ("agnews", "albert",  "eval_cls"),
    ("qqp",    "siamlog", "eval_nli"),
    ("qqp",    "albert",  "eval_nli"),
]

rows, index = [], []
for setting in ABLA_SETTINGS:
    row = []
    for ds, mdl, _ in ABLA_COLS:
        d = REPORTED / "abla-components" / f"clf-{ds}-{mdl}" / setting
        row.append(fmt(load_dir_stats(d)))
    rows.append(row)
    index.append(setting)

col_index = pd.MultiIndex.from_tuples([(ds, mdl) for ds, mdl, _ in ABLA_COLS], names=["dataset", "classifier"])
df_abla_components = pd.DataFrame(rows, index=pd.Index(index, name="setting"), columns=col_index)
df_abla_components


In [ ]:
# abla-kernels: rows=setting, cols=(dataset, classifier)
# settings: take (default) + each subdir of take-kernels/
KERNEL_SETTINGS = ["donttake", "last", "constant", "linear", "cosine", "take"]

def kernel_dir(base: Path, setting: str) -> Path:
    if setting == "take":
        return base / "take"
    return base / "take-kernels" / setting

rows, index = [], []
for setting in KERNEL_SETTINGS:
    row = []
    for ds, mdl, _ in ABLA_COLS:
        d = kernel_dir(REPORTED / "abla-kernels" / f"clf-{ds}-{mdl}", setting)
        row.append(fmt(load_dir_stats(d)))
    rows.append(row)
    index.append(setting)

df_abla_kernels = pd.DataFrame(rows, index=pd.Index(index, name="setting"), columns=col_index)
df_abla_kernels
